In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import numpy as np

In [ ]:
diabetes = load_diabetes(scaled=False)

# Создаем DataFrame для удобства
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='target')

print(f"Размерность X: {X.shape}")
print(f"Количество признаков: {len(diabetes.feature_names)}")
print(f"Признаки: {diabetes.feature_names}")
print(f"\nПервые 5 строк данных:")
X.head()

# Разделяем на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

In [ ]:
# Устанавливаем URI для хранения экспериментов MLflow
# Используем локальную директорию mlruns
mlflow.set_tracking_uri("file:./mlruns")

# Создаем или получаем эксперимент
experiment_name = "diabetes_prediction"
try:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location="./mlruns"
    )
except:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

print(f"ID эксперимента: {experiment_id}")

In [ ]:
# Создаем пайплайн с StandardScaler и RandomForestRegressor
pipeline = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
)

# Обучаем модель
pipeline.fit(X_train, y_train)

# Оцениваем качество на тестовой выборке
y_pred = pipeline.predict(X_test)
mse = np.mean((y_test - y_pred) ** 2)
r2 = pipeline.score(X_test, y_test)

print(f"Среднеквадратичная ошибка (MSE): {mse:.2f}")
print(f"Коэффициент детерминации (R²): {r2:.4f}")

In [ ]:
with mlflow.start_run(run_name="random_forest_diabetes"):
    # Логируем параметры модели
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("scaler", "StandardScaler")
    
    # Логируем метрики
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    
    # Логируем модель с именем "diabetes"
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name="diabetes"
    )
    
    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")

In [ ]:
# Получаем информацию о зарегистрированной модели
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "diabetes"

try:
    # Получаем последнюю версию модели
    model_versions = client.get_latest_versions(model_name, stages=["None"])
    if model_versions:
        latest_version = model_versions[0].version
        print(f"Последняя версия модели '{model_name}': {latest_version}")
        
        # Получаем run_id для этой версии
        model_version_details = client.get_model_version(model_name, latest_version)
        run_id = model_version_details.run_id
        print(f"Run ID для версии {latest_version}: {run_id}")
    else:
        print(f"Модель '{model_name}' не найдена")
except Exception as e:
    print(f"Ошибка при получении модели: {e}")

In [ ]:
# Загружаем модель из MLflow
try:
    # Загружаем последнюю версию модели
    model_uri = f"models:/{model_name}/latest"
    loaded_model = mlflow.sklearn.load_model(model_uri)
    print(f"Модель успешно загружена из {model_uri}")
    
    # Проверяем работу модели на тестовых данных
    test_pred = loaded_model.predict(X_test[:5])
    print(f"Пример предсказаний на тестовых данных: {test_pred}")
    
except Exception as e:
    print(f"Ошибка при загрузке модели: {e}")

In [ ]:
# Сохраняем информацию о признаках для использования в сервисе
feature_names = diabetes.feature_names
print("Признаки для входных данных:")
for i, feature in enumerate(feature_names):
    print(f"{i+1}. {feature}")

# Сохраняем пример статистики по признакам для проверки
feature_stats = X.describe()
print("\nСтатистика по признакам:")
feature_stats

# Сохраняем пример входных данных для тестирования
sample_input = X.iloc[0].to_dict()
print("Пример входных данных для API:")
print(sample_input)